In [0]:

# ---- CELL 1: config -------------------------------------------------
# Run `SHOW CATALOGS` in a scratch cell first if unsure.
# New Unity Catalog workspaces = "workspace"; older ones = "hive_metastore".
CATALOG = "workspace"
SCHEMA  = "cafe_karan"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_cafe_sales"

print("Target table:", BRONZE_TABLE)

# COMMAND ----------

# ---- CELL 2: create the schema --------------------------------------
# A schema is just a folder for tables. We do NOT create a catalog —
# most learner accounts lack permission, and it fails confusingly.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Schema ready")

# COMMAND ----------

# ---- CELL 3: find the CSV -------------------------------------------
# In a Databricks Git folder, the working directory IS the notebook's
# folder. The CSV sits right beside this notebook, so just the filename.
import os

CSV_PATH = os.path.join(os.getcwd(), "dirty_cafe_sales.csv")

print("Notebook folder:", os.getcwd())
print("CSV path:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))   # must be True before you go on

# COMMAND ----------

# ---- CELL 4: read it, everything as text -----------------------------
# inferSchema=False is the whole point of bronze. If Spark inferred types
# it would see "ERROR" in a number column and silently turn it into NULL,
# and you could never tell a blank apart from an ERROR afterwards.
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(CSV_PATH)
)

print("Rows read:", raw_df.count())     # expect 10000
raw_df.printSchema()                    # every column should say: string

# COMMAND ----------

# ---- CELL 5: look at it ---------------------------------------------
display(raw_df.limit(20))

# COMMAND ----------

# ---- CELL 6: fix the column names -----------------------------------
# Your headers have spaces ("Transaction ID"). Delta/Parquet CANNOT store
# column names containing  ,;{}()\n\t=  -- space included. Skip this and
# the write in cell 8 fails with "Attribute name contains invalid character(s)".
# toDF() renames every column at once, in order.
bronze_df = raw_df.toDF(*[c.strip().lower().replace(" ", "_") for c in raw_df.columns])

bronze_df.printSchema()

# COMMAND ----------

# ---- CELL 7: add ingestion metadata ---------------------------------
# Underscore prefix = "we added this, it wasn't in the source".
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    bronze_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

display(bronze_df.limit(5))

# COMMAND ----------

# ---- CELL 8: write the bronze table ---------------------------------
# mode("overwrite") makes this notebook safe to re-run. Without it you'd
# have 20,000 rows after the second run, 30,000 after the third.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(BRONZE_TABLE)
)

print("Written:", BRONZE_TABLE)

# COMMAND ----------

# ---- CELL 9: verify ---------------------------------------------------
bronze = spark.table(BRONZE_TABLE)
print("Rows in table:", bronze.count())   # must still be 10000
display(bronze.limit(10))

# COMMAND ----------

# ---- CELL 10: how dirty is it? (measure only, change nothing) --------
from pyspark.sql.functions import when, sum as _sum, count, lit
from functools import reduce

DATA_COLS = [c for c in bronze.columns if not c.startswith("_")]

def is_dirty(c):
    return col(c).isin("ERROR", "UNKNOWN") | (col(c) == "") | col(c).isNull()

# dirty count per column
display(bronze.select(*[
    _sum(when(is_dirty(c), 1).otherwise(0)).alias(c) for c in DATA_COLS
]))

# COMMAND ----------

# ---- CELL 11: how many rows are perfectly clean? ---------------------
row_is_dirty = reduce(lambda a, b: a | b, [is_dirty(c) for c in DATA_COLS])

display(bronze.select(
    count(lit(1)).alias("total_rows"),
    _sum(when(row_is_dirty, 0).otherwise(1)).alias("fully_clean_rows"),
    _sum(when(row_is_dirty, 1).otherwise(0)).alias("rows_with_dirt"),
))
